## Build two tower model

In [1]:
!pwd

/home/jupyter/import-backup/crispy_towers/notebooks


In [2]:
import sys
sys.path.append("..")
import env_config

print(f"PREFIX: {env_config.PREFIX}")
print(f"PROJECT_ID: {env_config.PROJECT_ID}")
print(f"LOCATION: {env_config.LOCATION}")

PREFIX: jt-towers-v1
PROJECT_ID: hybrid-vertex
LOCATION: us-central1


## imports

In [3]:
import os
import json
import time
from pprint import pprint
import pickle as pkl
import numpy as np
import pandas as pd

import logging
logging.disable(logging.WARNING)

import warnings
warnings.filterwarnings('ignore')

# tensorflow
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
import tensorflow_recommenders as tfrs

# GPU
import gc
from numba import cuda

# google cloud
from google.cloud import aiplatform, storage
storage_client = storage.Client(project=env_config.PROJECT_ID)
aiplatform.init(project=env_config.PROJECT_ID, location=env_config.LOCATION)

# this repo
sys.path.append("..")
from src.data import data_utils as data_utils
from src.model import two_tower
from src.model import train_utils

print(f"pandas version = {pd.__version__}")
print(f"numpy version = {np.__version__}")
print(f"Vertex AI SDK version = {aiplatform.__version__}")
print(f"Cloud Storage version = {storage.__version__}")
print(f"Tensorflow version = {tf.__version__}")
# print(f"TFRS version = {tfrs.__version__}")

pandas version = 2.2.3
numpy version = 1.26.4
Vertex AI SDK version = 1.60.0
Cloud Storage version = 2.19.0
Tensorflow version = 2.11.0


### detect GPU

In [5]:
device = cuda.get_current_device()
device.reset()
gc.collect()

14

In [6]:
print(f"Num GPUs avail : {len(tf.config.list_physical_devices('GPU'))}")
print(f"Device name    : {device.name.decode()}")

Num GPUs avail : 1
Device name    : NVIDIA A100-SXM4-40GB


## Get data

In [7]:
options = tf.data.Options()
options.experimental_distribute.auto_shard_policy = tf.data.experimental.AutoShardPolicy.AUTO

In [8]:
GCS_DATA_PATH = f"{env_config.BUCKET_URI}/{env_config.EXAMPLE_GEN_GCS_PATH}"
print(f"GCS_DATA_PATH : {GCS_DATA_PATH}")

! gsutil ls $GCS_DATA_PATH

GCS_DATA_PATH : gs://jt-towers-v1-hybrid-vertex-bucket/data/movielens/m1m_v2
gs://jt-towers-v1-hybrid-vertex-bucket/data/movielens/m1m_v2/candidates/
gs://jt-towers-v1-hybrid-vertex-bucket/data/movielens/m1m_v2/train/
gs://jt-towers-v1-hybrid-vertex-bucket/data/movielens/m1m_v2/val/
gs://jt-towers-v1-hybrid-vertex-bucket/data/movielens/m1m_v2/vocabs/


### train records

In [9]:
train_files = []

for blob in storage_client.list_blobs(
    f"{env_config.BUCKET_NAME}", 
    prefix=f'{env_config.EXAMPLE_GEN_GCS_PATH}/train/', 
    # delimiter='/'
):
    if '.tfrecord' in blob.name:
        train_files.append(blob.public_url.replace("https://storage.googleapis.com/", "gs://"))
        
mv_dataset = tf.data.TFRecordDataset(train_files[0])
mv_dataset = mv_dataset.map(data_utils._parse_function)

# for x in mv_dataset.batch(1).take(1):
#     pprint(x)

### val records

In [10]:
val_files = []

for blob in storage_client.list_blobs(
    f"{env_config.BUCKET_NAME}", 
    prefix=f'{env_config.EXAMPLE_GEN_GCS_PATH}/val/', 
    # delimiter='/'
):
    if '.tfrecord' in blob.name:
        val_files.append(blob.public_url.replace("https://storage.googleapis.com/", "gs://"))
        
val_dataset = tf.data.TFRecordDataset(val_files[0])
val_dataset = val_dataset.map(data_utils._parse_function)

# for x in val_dataset.batch(1).take(1):
#     pprint(x)

### Candidate dataset

In [11]:
CANDIDATE_FILES = [
    "gs://jt-towers-v1-hybrid-vertex-bucket/data/movielens/m1m_v2/candidates/train/output-00000-of-00002.tfrecord",
    "gs://jt-towers-v1-hybrid-vertex-bucket/data/movielens/m1m_v2/candidates/train/output-00001-of-00002.tfrecord"
]

# candidate_dataset = tf.data.TFRecordDataset(CANDIDATE_FILES)
# parsed_candidate_dataset = candidate_dataset.map(data_utils._parse_candidates_fn)

candidate_dataset = tf.data.Dataset.from_tensor_slices(CANDIDATE_FILES)

parsed_candidate_dataset = candidate_dataset.interleave(
    data_utils.full_parse,
    cycle_length=tf.data.AUTOTUNE, 
    num_parallel_calls=tf.data.AUTOTUNE,
    deterministic=False
).map(
    data_utils._parse_candidates_fn, 
    num_parallel_calls=tf.data.AUTOTUNE
).with_options(
    options
)
parsed_candidate_dataset = parsed_candidate_dataset.cache()

for z in parsed_candidate_dataset.batch(1).take(1):
    pprint(z)

{'target_movie_id': <tf.Tensor: shape=(1,), dtype=string, numpy=array([b'2522'], dtype=object)>,
 'target_movie_title': <tf.Tensor: shape=(1,), dtype=string, numpy=array([b"Airport '77 (1977)"], dtype=object)>,
 'target_movie_year': <tf.Tensor: shape=(1,), dtype=int64, numpy=array([1977])>}


### vocab file

In [12]:
VOCAB_FILENAME="vocab_dict.pkl"

In [13]:
EXISTING_VOCAB_FILE = f'gs://{env_config.BUCKET_NAME}/{env_config.EXAMPLE_GEN_GCS_PATH}/vocabs/{VOCAB_FILENAME}'
print(f"Downloading vocab...")

os.system(f'gsutil -q cp {EXISTING_VOCAB_FILE} .')
print(f"Downloaded vocab from: {EXISTING_VOCAB_FILE}\n")

filehandler = open(VOCAB_FILENAME, 'rb')
vocab_dict = pkl.load(filehandler)
filehandler.close()

# for key in vocab_dict.keys():
#     pprint(key)

Downloaded vocab from: gs://jt-towers-v1-hybrid-vertex-bucket/data/movielens/m1m_v2/vocabs/vocab_dict.pkl



# Build and compile two-tower model

In [14]:
USE_CROSS_LAYER = True
USE_DROPOUT = True
SEED = 1234
EMBEDDING_DIM = 256
PROJECTION_DIM = int(EMBEDDING_DIM / 4) # 50  
SEED = 1234
DROPOUT_RATE = 0.33
MAX_TOKENS = 20000
LAYER_SIZES=[EMBEDDING_DIM*4,EMBEDDING_DIM*2,EMBEDDING_DIM]

BATCH_SIZE = 1024

LR = .1
opt = tf.keras.optimizers.Adagrad(LR)

print(f"EMBEDDING_DIM  : {EMBEDDING_DIM}")
print(f"PROJECTION_DIM : {PROJECTION_DIM}")
print(f"DROPOUT_RATE   : {DROPOUT_RATE}")
print(f"MAX_TOKENS     : {MAX_TOKENS}")
print(f"LAYER_SIZES    : {LAYER_SIZES}")
print(f"BATCH_SIZE     : {BATCH_SIZE}")
print(f"LR             : {LR}")

EMBEDDING_DIM  : 256
PROJECTION_DIM : 64
DROPOUT_RATE   : 0.33
MAX_TOKENS     : 20000
LAYER_SIZES    : [1024, 512, 256]
BATCH_SIZE     : 1024
LR             : 0.1


In [15]:
model = two_tower.TheTwoTowers(
    layer_sizes=LAYER_SIZES, 
    vocab_dict=vocab_dict, 
    parsed_candidate_dataset=parsed_candidate_dataset,
    embedding_dim=EMBEDDING_DIM,
    projection_dim=PROJECTION_DIM,
    seed=SEED,
    use_cross_layer=USE_CROSS_LAYER,
    use_dropout=USE_DROPOUT,
    dropout_rate=DROPOUT_RATE,
    max_tokens=MAX_TOKENS,
    max_context_length=env_config.MAX_CONTEXT_LENGTH,
    max_genre_length=env_config.MAX_GENRE_LENGTH,
    compute_batch_metrics=False
)

In [16]:
model.compile(optimizer=opt)

model

## inspect model layers

In [17]:
## Quick look at the layers
print("User (query) Tower:")

for i, l in enumerate(model.query_tower.layers):
    print(i, l.name)

User (query) Tower:
0 user_id_emb_model
1 user_gender_emb_model
2 user_age_emb_model
3 user_occ_emb_model
4 user_zip_emb_model
5 context_mv_id_emb_model
6 context_mv_rating_emb_model
7 context_rating_ts_emb_model
8 context_mv_year_emb_model
9 context_mv_title_emb_model
10 query_cross_layer
11 query_dense_layers


In [18]:
print("Track (candidate) Tower:")
for i, l in enumerate(model.candidate_tower.layers):
    print(i, l.name)

Track (candidate) Tower:
0 target_mv_id_emb_model
1 target_mv_year_emb_model
2 target_mv_title_emb_model
3 candidate_cross_layer
4 candidate_dense_layers


# Train two-tower model

## Data input pipelines

In [19]:
# train_dataset    
train_dataset = tf.data.Dataset.from_tensor_slices(train_files).prefetch(
    tf.data.AUTOTUNE,
)
train_dataset = train_dataset.interleave(
    data_utils.full_parse,
    cycle_length=tf.data.AUTOTUNE, 
    num_parallel_calls=tf.data.AUTOTUNE,
    deterministic=False,
).map(
    data_utils._parse_function,
    num_parallel_calls=tf.data.AUTOTUNE
).batch(
    BATCH_SIZE 
).prefetch(
    tf.data.AUTOTUNE,
).with_options(
    options
)
train_dataset

<_OptionsDataset element_spec={'context_movie_genre': TensorSpec(shape=(None, 10), dtype=tf.string, name=None), 'context_movie_id': TensorSpec(shape=(None, 10), dtype=tf.string, name=None), 'context_movie_rating': TensorSpec(shape=(None, 10), dtype=tf.float32, name=None), 'context_movie_title': TensorSpec(shape=(None, 10), dtype=tf.string, name=None), 'context_movie_year': TensorSpec(shape=(None, 10), dtype=tf.int64, name=None), 'context_rating_timestamp': TensorSpec(shape=(None, 10), dtype=tf.int64, name=None), 'target_movie_id': TensorSpec(shape=(None,), dtype=tf.string, name=None), 'target_movie_rating': TensorSpec(shape=(None,), dtype=tf.float32, name=None), 'target_movie_title': TensorSpec(shape=(None,), dtype=tf.string, name=None), 'target_movie_year': TensorSpec(shape=(None,), dtype=tf.int64, name=None), 'target_rating_timestamp': TensorSpec(shape=(None,), dtype=tf.int64, name=None), 'user_age': TensorSpec(shape=(None,), dtype=tf.int64, name=None), 'user_gender': TensorSpec(shap

In [20]:
# valid_dataset
val_ds = tf.data.Dataset.from_tensor_slices(val_files)

valid_dataset = val_ds.prefetch(
    tf.data.AUTOTUNE,
).interleave(
    data_utils.full_parse,
    num_parallel_calls=tf.data.AUTOTUNE,
    cycle_length=tf.data.AUTOTUNE, 
    deterministic=False,
).map(
    data_utils._parse_function, 
    num_parallel_calls=tf.data.AUTOTUNE
).batch(
    BATCH_SIZE
).prefetch(
    tf.data.AUTOTUNE,
).with_options(
    options
)
# valid_dataset

## Vertex Experiments

In [22]:
EXP_VERSION = "v1"

In [23]:
EXPERIMENT_NAME = f'local-towers-{EXP_VERSION}'

invoke_time = time.strftime("%Y%m%d-%H%M%S")
RUN_NAME = f'run-{invoke_time}'

EXPERIMENT_DIR  = os.path.join(env_config.BUCKET_URI, EXPERIMENT_NAME)
BASE_OUTPUT_DIR = os.path.join(EXPERIMENT_DIR, RUN_NAME)
LOG_DIR         = os.path.join(BASE_OUTPUT_DIR, "logs")
CHECKPT_DIR     = os.path.join(BASE_OUTPUT_DIR, "chkpoint")
ARTIFACTS_DIR   = os.path.join(BASE_OUTPUT_DIR, "artifacts")

QUERY_TOWER_DIR = os.path.join(ARTIFACTS_DIR, 'query_tower')
CAND_TOWER_DIR  = os.path.join(ARTIFACTS_DIR, 'candidate_tower')

aiplatform.init(
    project=env_config.PROJECT_ID,
    location=env_config.LOCATION,
    experiment=EXPERIMENT_NAME
)

print(f"EXPERIMENT_NAME   : {EXPERIMENT_NAME}")
print(f"RUN_NAME          : {RUN_NAME}\n")

print(f"EXPERIMENT_DIR    : {EXPERIMENT_DIR}")
print(f"BASE_OUTPUT_DIR   : {BASE_OUTPUT_DIR}\n")

print(f"LOG_DIR           : {LOG_DIR}")
print(f"CHECKPT_DIR       : {CHECKPT_DIR}")
print(f"ARTIFACTS_DIR     : {ARTIFACTS_DIR}")
print(f"QUERY_TOWER_DIR   : {QUERY_TOWER_DIR}")
print(f"CAND_TOWER_DIR    : {CAND_TOWER_DIR}\n")

EXPERIMENT_NAME   : local-towers-v1
RUN_NAME          : run-20250129-085402

EXPERIMENT_DIR    : gs://jt-towers-v1-hybrid-vertex-bucket/local-towers-v1
BASE_OUTPUT_DIR   : gs://jt-towers-v1-hybrid-vertex-bucket/local-towers-v1/run-20250129-085402

LOG_DIR           : gs://jt-towers-v1-hybrid-vertex-bucket/local-towers-v1/run-20250129-085402/logs
CHECKPT_DIR       : gs://jt-towers-v1-hybrid-vertex-bucket/local-towers-v1/run-20250129-085402/chkpoint
ARTIFACTS_DIR     : gs://jt-towers-v1-hybrid-vertex-bucket/local-towers-v1/run-20250129-085402/artifacts
QUERY_TOWER_DIR   : gs://jt-towers-v1-hybrid-vertex-bucket/local-towers-v1/run-20250129-085402/artifacts/query_tower
CAND_TOWER_DIR    : gs://jt-towers-v1-hybrid-vertex-bucket/local-towers-v1/run-20250129-085402/artifacts/candidate_tower



### Tensorboard Summary Writer

In [24]:
# train_summary_writer = tf.compat.v2.summary.create_file_writer(
#     f"{LOG_DIR}", flush_millis=10 * 1000
# )

# train_summary_writer.set_as_default()

# # Create new TB instance
# TENSORBOARD_DISPLAY_NAME=f"{EXPERIMENT_NAME}"
# tensorboard = vertex_ai.Tensorboard.create(display_name=TENSORBOARD_DISPLAY_NAME)
# TB_RESOURCE_NAME = tensorboard.resource_name

# print(f"TB_RESOURCE_NAME: {TB_RESOURCE_NAME}")

## Train model

In [25]:
NUM_EPOCHS = 10
VALID_FREQ = NUM_EPOCHS+1
HIST_FREQ = 0
EMBED_FREQ = 0

os.environ['TF_GPU_ALLOCATOR'] ='cuda_malloc_async'
os.environ['TF_GPU_THREAD_MODE'] = 'gpu_private'
os.environ['TF_GPU_THREAD_COUNT'] = '1'
# os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# =================================
# callbacks
# =================================
tensorboard_callback = tf.keras.callbacks.TensorBoard(
    log_dir=LOG_DIR, 
    histogram_freq=HIST_FREQ, 
    write_graph=True,
    # profile_batch=(20,30),
    # embeddings_freq=EMBED_FREQ,
    # embeddings_metadata=LOCAL_EMB_FILE
)

# model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
#     filepath=CHECKPT_DIR + "/cp-{epoch:03d}-loss={loss:.2f}.ckpt",
#     save_weights_only=True,
#     save_best_only=True,
#     monitor='total_loss',
#     mode='min',
#     save_freq='epoch',
#     verbose=0,
# )

In [26]:
# =================================
# train loop
# =================================
start_time = time.time()

layer_history = model.fit(
    x=train_dataset,
    # validation_data=valid_dataset,
    # validation_freq=VALID_FREQ,
    epochs=NUM_EPOCHS,
    steps_per_epoch=None, # 75
    # validation_steps=100,
    callbacks=[
        tensorboard_callback,
        # model_checkpoint_callback
    ],
    verbose=1
)

end_time = time.time()
val_keys = [v for v in layer_history.history.keys()]
runtime_mins = int((end_time - start_time) / 60)

print(f"total runtime : {runtime_mins}")

Epoch 1/10
874/874 [==============================] - 88s 92ms/step - batch_cat_acc@10: 0.0102 - batch_cat_acc@50: 0.0493 - factorized_top_k/top_10_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_50_categorical_accuracy: 0.0000e+00 - loss: 7098.1816 - regularization_loss: 0.0000e+00 - total_loss: 7098.1816
Epoch 2/10
874/874 [==============================] - 43s 49ms/step - batch_cat_acc@10: 0.0103 - batch_cat_acc@50: 0.0496 - factorized_top_k/top_10_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_50_categorical_accuracy: 0.0000e+00 - loss: 7093.8766 - regularization_loss: 0.0000e+00 - total_loss: 7093.8766
Epoch 3/10
874/874 [==============================] - 40s 45ms/step - batch_cat_acc@10: 0.0124 - batch_cat_acc@50: 0.0586 - factorized_top_k/top_10_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_50_categorical_accuracy: 0.0000e+00 - loss: 7057.8781 - regularization_loss: 0.0000e+00 - total_loss: 7057.8781
Epoch 4/10
874/874 [==============================] 

In [27]:
# gather the metrics for the last epoch to be saved in metrics
metrics_dict = {"runtime": runtime_mins}

_ = [metrics_dict.update({key: layer_history.history[key][-1]}) for key in val_keys]

metrics_dict

{'runtime': 7,
 'batch_cat_acc@10': 0.233541801571846,
 'batch_cat_acc@50': 0.5499356389045715,
 'factorized_top_k/top_10_categorical_accuracy': 0.0,
 'factorized_top_k/top_50_categorical_accuracy': 0.0,
 'loss': 3994.905029296875,
 'regularization_loss': 0,
 'total_loss': 3994.905029296875}

### TensorBoard

In [28]:
# from tensorboard import notebook
# notebook.list() # View open TensorBoard instances

In [29]:
!gsutil ls $LOG_DIR

gs://jt-towers-v1-hybrid-vertex-bucket/local-towers-v1/run-20250129-085402/logs/
gs://jt-towers-v1-hybrid-vertex-bucket/local-towers-v1/run-20250129-085402/logs/train/


In [33]:
# %load_ext tensorboard
# %reload_ext tensorboard

In [32]:
# %tensorboard --logdir=$LOG_DIR

### Save each tower

In [34]:
# save query tower
tf.saved_model.save(
    model.query_tower, export_dir=QUERY_TOWER_DIR
)

# save candidate tower
tf.saved_model.save(
    model.candidate_tower, export_dir=CAND_TOWER_DIR
)

In [35]:
!gsutil ls $ARTIFACTS_DIR

gs://jt-towers-v1-hybrid-vertex-bucket/local-towers-v1/run-20250129-085402/artifacts/
gs://jt-towers-v1-hybrid-vertex-bucket/local-towers-v1/run-20250129-085402/artifacts/candidate_tower/
gs://jt-towers-v1-hybrid-vertex-bucket/local-towers-v1/run-20250129-085402/artifacts/query_tower/


In [36]:
!gsutil ls $QUERY_TOWER_DIR

gs://jt-towers-v1-hybrid-vertex-bucket/local-towers-v1/run-20250129-085402/artifacts/query_tower/
gs://jt-towers-v1-hybrid-vertex-bucket/local-towers-v1/run-20250129-085402/artifacts/query_tower/fingerprint.pb
gs://jt-towers-v1-hybrid-vertex-bucket/local-towers-v1/run-20250129-085402/artifacts/query_tower/saved_model.pb
gs://jt-towers-v1-hybrid-vertex-bucket/local-towers-v1/run-20250129-085402/artifacts/query_tower/assets/
gs://jt-towers-v1-hybrid-vertex-bucket/local-towers-v1/run-20250129-085402/artifacts/query_tower/variables/


# Evaluate Model

In [37]:
start_time = time.time()

eval_dict_v1 = model.evaluate(
    x=valid_dataset,
    verbose="auto",
    return_dict=True
)

end_time = time.time()

elapsed_mins = int((end_time - start_time) / 60)
print(f"elapsed_mins: {elapsed_mins}")

98/98 [==============================] - 34s 323ms/step - batch_cat_acc@10: 0.2082 - batch_cat_acc@50: 0.5080 - factorized_top_k/top_10_categorical_accuracy: 0.0914 - factorized_top_k/top_50_categorical_accuracy: 0.2954 - loss: 5508.6724 - regularization_loss: 0.0000e+00 - total_loss: 5508.6724
elapsed_mins: 0


In [38]:
eval_dict_v1

{'batch_cat_acc@10': 0.20815353095531464,
 'batch_cat_acc@50': 0.50799161195755,
 'factorized_top_k/top_10_categorical_accuracy': 0.09140288084745407,
 'factorized_top_k/top_50_categorical_accuracy': 0.29543235898017883,
 'loss': 244.70191955566406,
 'regularization_loss': 0,
 'total_loss': 244.70191955566406}

## efficient eval w/ ScaNN

* approximate with scann

In [39]:
start_time = time.time()

scann = tfrs.layers.factorized_top_k.ScaNN(
    num_reordering_candidates=500,
    num_leaves_to_search=30
)

scann.index_from_dataset(
    candidates=parsed_candidate_dataset.batch(128).cache().map(
        lambda x: (
            x['target_movie_id'], 
            model.candidate_tower(x)
        )
    )
)

end_time = time.time()

elapsed_scann_mins = int((end_time - start_time) / 60)
print(f"elapsed_scann_mins: {elapsed_scann_mins}")

elapsed_scann_mins: 0


In [40]:
start_time = time.time()

model.task.factorized_metrics = tfrs.metrics.FactorizedTopK(
    candidates=scann
)
model.compile()

scann_result = model.evaluate(
    x=valid_dataset, 
    return_dict=True, 
    verbose=1,
)

end_time = time.time()

elapsed_scann_eval_mins = int((end_time - start_time) / 60)
print(f"elapsed_scann_eval_mins: {elapsed_scann_eval_mins}")

98/98 [==============================] - 7s 59ms/step - batch_cat_acc@10: 0.2077 - batch_cat_acc@50: 0.5087 - factorized_top_k/top_1_categorical_accuracy: 0.0103 - factorized_top_k/top_5_categorical_accuracy: 0.0496 - factorized_top_k/top_10_categorical_accuracy: 0.0910 - factorized_top_k/top_50_categorical_accuracy: 0.2921 - factorized_top_k/top_100_categorical_accuracy: 0.4186 - loss: 5509.8836 - regularization_loss: 0.0000e+00 - total_loss: 5509.8836
elapsed_scann_eval_mins: 0


In [41]:
scann_result

{'batch_cat_acc@10': 0.20772100985050201,
 'batch_cat_acc@50': 0.508705735206604,
 'factorized_top_k/top_1_categorical_accuracy': 0.010310107842087746,
 'factorized_top_k/top_5_categorical_accuracy': 0.04961927980184555,
 'factorized_top_k/top_10_categorical_accuracy': 0.09100053459405899,
 'factorized_top_k/top_50_categorical_accuracy': 0.29208284616470337,
 'factorized_top_k/top_100_categorical_accuracy': 0.4186004400253296,
 'loss': 280.82403564453125,
 'regularization_loss': 0,
 'total_loss': 280.82403564453125}

# Save the candidate embeddings

> These will be the files we use for the index

## compute embeddings

In [42]:
start_time = time.time()

candidate_embeddings = parsed_candidate_dataset.batch(128).map(
    lambda x: (
        x['target_movie_id'],
        train_utils.tf_if_null_return_zero(
            model.candidate_tower(x)
        )
    )
)

elapsed_mins = int((time.time() - start_time) / 60)
print(f"elapsed_mins: {elapsed_mins}")

elapsed_mins: 0


In [43]:
embs = []
for emb in candidate_embeddings:
    embs.append(emb)
    
print(f"Length of embs: {len(embs)}")

Length of embs: 31


### clean embeddings

In [44]:
start_time = time.time()

cleaned_embs = [] #clean up the output
movie_ids = []

for ids , embedding in embs:
    cleaned_embs.extend(embedding.numpy())
    movie_ids.extend(ids.numpy())

end_time = time.time()
elapsed_time = int((end_time - start_time) / 60)
print(f"elapsed_time: {elapsed_time}")

elapsed_time: 0


In [45]:
print(f"Length of cleaned_embs: {len(cleaned_embs)}")
print(f"Length of movie_ids: {len(movie_ids)}")
cleaned_embs[0]

Length of cleaned_embs: 3883
Length of movie_ids: 3883


array([-11.088015  ,  -6.9466023 ,  10.566695  ,  10.762018  ,
         3.5954878 , -10.408398  , -21.263275  ,  -0.24931967,
        -6.918656  ,  -3.3199866 ,  -2.504135  ,  -7.0435643 ,
       -12.070675  ,  -7.828643  ,  12.877644  , -11.557793  ,
         5.698592  ,   9.483826  ,  13.273245  ,   9.787577  ,
        -6.29685   , -11.897749  ,   6.6992807 ,   4.9070387 ,
       -10.250976  ,  -2.3414044 ,   4.701572  ,   5.3818316 ,
         8.251281  ,  -6.670981  ,   5.659448  , -10.593693  ,
        -3.5893931 ,  12.998872  ,  11.730904  ,  10.134711  ,
        -7.0735383 ,   7.645147  ,   7.6223803 ,   7.966808  ,
        -3.8121643 ,  -3.0588484 ,   5.494987  , -12.297205  ,
        -6.7805    ,   6.554025  ,   3.9097483 , -12.31805   ,
        12.13267   ,  -5.753066  , -11.061449  ,   1.209765  ,
        -5.6752524 ,   6.5051985 ,  -5.069338  ,   6.250127  ,
        -9.854443  ,  12.137005  ,   4.7524333 ,  11.9844265 ,
        -7.9133954 ,   5.6456747 ,  -4.228604  ,  -2.99

In [46]:
movie_ids_decoded = [z.decode("utf-8") for z in movie_ids]

print(f"Length of movie_ids_decoded: {len(movie_ids_decoded)}")
movie_ids_decoded[0]

Length of movie_ids_decoded: 3883


'3836'

In [47]:
print(f"Length of movie_ids       : {len(movie_ids)}")
print(f"Length of clean movie_ids : {len(movie_ids_decoded)}")

Length of movie_ids       : 3883
Length of clean movie_ids : 3883


### check for NaNs

In [48]:
start_time = time.time()

bad_records = []

for i, emb in enumerate(cleaned_embs):
    bool_emb = np.isnan(emb)
    for val in bool_emb:
        if val:
            bad_records.append(i)

end_time = time.time()
elapsed_time = int((end_time - start_time) / 60)
print(f"elapsed_time: {elapsed_time}")

bad_record_filter = np.unique(bad_records)

print(f"bad_records: {len(bad_records)}")
print(f"bad_record_filter: {len(bad_record_filter)}")

elapsed_time: 0
bad_records: 0
bad_record_filter: 0


In [49]:
start_time = time.time()

movie_ids_valid = []
emb_valid = []

for i, pair in enumerate(zip(movie_ids_decoded, cleaned_embs)):
    if i in bad_record_filter:
        pass
    else:
        m_id, embed = pair
        movie_ids_valid.append(m_id)
        emb_valid.append(embed)
        
end_time = time.time()
elapsed_time = int((end_time - start_time) / 60)
print(f"elapsed_time: {elapsed_time}")

print(f"num embeddings : {len(emb_valid)}")
print(f"num movie IDs  : {len(movie_ids_valid)}")

elapsed_time: 0
num embeddings : 3883
num movie IDs  : 3883


## write embeddings to json

Save to the required format
> make sure you start out with a clean empty file for the append write

In [52]:
CANDIDATE_EMB_JSON = 'candidate_embeddings.json'

!rm $CANDIDATE_EMB_JSON > /dev/null

!touch $CANDIDATE_EMB_JSON

In [53]:
with open(f'{CANDIDATE_EMB_JSON}', 'w') as f:
    for mv_id, emb in zip(movie_ids_valid, emb_valid):
        f.write('{"id":"' + str(mv_id) + '",')
        f.write('"embedding":[' + ",".join(str(x) for x in list(emb)) + "]}")
        f.write("\n")

### save json to Cloud Storage

In [54]:
from google.cloud.storage.bucket import Bucket
from google.cloud.storage.blob import Blob

In [55]:
INDEX_GCS_URI = os.path.join(ARTIFACTS_DIR, "candidate-embeddings")

DESTINATION_BLOB_NAME = f'candidate_embeddings_{EMBEDDING_DIM}.json'
SOURCE_FILE_NAME = CANDIDATE_EMB_JSON

print(f"INDEX_GCS_URI         : {INDEX_GCS_URI}")
print(f"DESTINATION_BLOB_NAME : {DESTINATION_BLOB_NAME}")
print(f"SOURCE_FILE_NAME      : {SOURCE_FILE_NAME}")

INDEX_GCS_URI         : gs://jt-towers-v1-hybrid-vertex-bucket/local-towers-v1/run-20250129-085402/artifacts/candidate-embeddings
DESTINATION_BLOB_NAME : candidate_embeddings_256.json
SOURCE_FILE_NAME      : candidate_embeddings.json


In [56]:
blob = Blob.from_string(os.path.join(INDEX_GCS_URI, DESTINATION_BLOB_NAME))
blob.bucket._client = storage_client
blob.upload_from_filename(SOURCE_FILE_NAME)

**finished**